### Compares FAISS (exact search) and Annoy (approximate search) 
Performing semantic similarity search on embeddings and measuring performance. Highlights trade-off between accuracy (FAISS) and speed/scalability (Annoy) using real query results and timing.

In [ ]:
import numpy as np
import faiss
from annoy import AnnoyIndex
from sentence_transformers import SentenceTransformer
import time

In [ ]:
def faiss_search(embeddings, query_vector, k):
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    distances, indices = index.search(np.array([query_vector]), k)
    return distances[0], indices[0]

def annoy_search(embeddings, query_vector, k):
    dimension = embeddings.shape[1]
    index = AnnoyIndex(dimension, 'angular')
    for i, embedding in enumerate(embeddings):
        index.add_item(i, embedding)
    index.build(10)  # 10 trees - more trees gives higher precision when querying
    indices, distances = index.get_nns_by_vector(query_vector, k, include_distances=True)
    return distances, indices


In [ ]:
# Load a pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("SentenceTransformer model loaded.")

# Sample custom data (text)
texts = [
    "FAISS is a library for efficient similarity search.",
    "Vectors represent data in numerical form.",
    "Embedding models convert text to vectors.",
    "Local vector databases can be faster for small datasets.",
    "FAISS supports both CPU and GPU operations.",
    "Annoy is an efficient library for approximate nearest neighbor search.",
    "Vector databases are crucial for large-scale machine learning applications.",
    "Similarity search is used in recommendation systems and information retrieval.",
    "Dimensionality reduction techniques can improve search efficiency.",
    "Cosine similarity is a common metric in vector space models."
]

# Convert texts to vectors
embeddings = model.encode(texts)
print(f"Converted {len(texts)} texts to embeddings.")

# Define vector dimension based on the model's output
dimension = embeddings.shape[1]
print(f"Vector dimension: {dimension}")


In [ ]:
# Example query
query_text = "Nature is Beautiful"
query_vector = model.encode([query_text])[0]
print(f"Encoded query: '{query_text}'")
print(f"Query vector: {query_vector}")

In [ ]:
# Perform the query with FAISS
print("\nFAISS Results:")
start_time = time.time()
faiss_distances, faiss_indices = faiss_search(embeddings, query_vector, 3)
faiss_time = time.time() - start_time
for i, (distance, idx) in enumerate(zip(faiss_distances, faiss_indices)):
    print(f"Rank: {i+1}")
    print(f"Text: {texts[idx]}")
    print(f"Distance: {distance}")
    print()
print(f"FAISS search time: {faiss_time:.6f} seconds")


In [ ]:
    # Perform the query with Annoy
print("\nAnnoy Results:")
start_time = time.time()
annoy_distances, annoy_indices = annoy_search(embeddings, query_vector, 3)
annoy_time = time.time() - start_time
for i, (distance, idx) in enumerate(zip(annoy_distances, annoy_indices)):
    print(f"Rank: {i+1}")
    print(f"Text: {texts[idx]}")
    print(f"Distance: {distance}")
    print()
print(f"Annoy search time: {annoy_time:.6f} seconds")